# PINN Failure Diagnosis and Targeted Remedies for Zonal-Jet Dynamics
**Project 3 — PINN/PINO in the Field**  
**Optiforge — Team 8**  
**Course:** Introduction to Optimization for Mechanical Engineers  
**Class:** [MAE 598]  
**Instructor:** [Instructor Yi Ren]  
**Team Members:**  
[Romeo Anderson Konadu]·[Yen-Chun Pan]·[Wei Lu]·[Aaldin Kesavan Helen]  · [Kouassi Ngoran Abel] 

**Arizona State University**  
**Fall 2026**

## Interview
Our team contacted Professor Huei-Ping Huang because his research includes
geophysical fluid dynamics, atmospheric and ocean simulation, and zonal jets.
We initially approached him after reading Huang and Robinson (1998),
Two-Dimensional Turbulence and Persistent Zonal Jets in a Global Barotropic
Model. We met with him during his office hours on October 14 and discussed the
two-dimensional PDE and its physical context. However, after the first meeting
we realized that we had not defined a sufficiently clear or manageable PINN
problem. After reviewing the model more carefully that evening, we concluded
that it was not a suitable minimal case for the time and computing resources
available for this project.

We therefore returned to Professor Huang's office hours on October 15,
explained our concern, and asked whether the physical problem could be reduced
without removing its essential behavior. He suggested the related study by
Manfroi and Young (1999), which begins with two-dimensional barotropic dynamics
on a beta plane and derives a reduced amplitude equation. The derivation uses
scale separation and a perturbation/multiple-scale expansion slightly above
the instability threshold; restricting the result to zonally uniform
solutions then produces the one-dimensional PDE used in this project.
Professor Huang explained the physical and mathematical idea behind this
reduction, but he did not formally review or confirm our final problem brief or
our interpretation of the PINN experiment.

This gave us a tractable equation that remained connected to his research on
zonal-jet dynamics. Because these studies predated the widespread use of PINNs
and did not apply a PINN to this problem, they also gave us a clear opportunity
to test whether physics-informed learning is useful here.
Zonal jets are persistent east-west flows observed in rotating atmospheres and
oceans. Full turbulence simulations are computationally demanding and contain
many interacting spatial and temporal scales. Manfroi and Young (1999) derived
a reduced one-dimensional amplitude equation that retains nonlinear jet
formation, scale selection, and slow jet merger. It is therefore a useful
minimal case for testing whether physics-informed neural networks can recover
long-time dynamics governed by a nonlinear, fourth-order PDE.

## Problem Brief
### Paper's Topic
In their foundational study, Manfroi & Young (1999) investigate the hydrodynamic stability and non-linear spatio-temporal evolution of forced barotropic sinusoidal shear flow on a $\beta$-plane subject to bottom drag and lateral viscosity. In geophysical fluid dynamics, small-scale turbulent forcing or deterministic stirring drives an energy transfer that reorganizes flow into persistent, large-scale zonal jets. A defining characteristic of these geostrophic velocity fields is a strong structural asymmetry: fast, narrow eastward jets separated by broader, slower westward flows.

Without bottom friction, the system undergoes an unmitigated one-dimensional inverse cascade where neighboring westward flows merge by absorbing the eastward jets separating them, ultimately driving the system toward a single domain-scale jet. The introduction of bottom drag arrests this scale growth, establishing a stable, periodic array of zonal jets whose asymptotic spacing scales inversely with the drag coefficient.

### Computational Difficulties in Paper
While traditional numerical schemes (such as Fourier pseudo-spectral solvers) compute forward trajectories rapidly and achieve machine precision, their primary difficulty in modeling beta-plane jet dynamics stems from the extremely slow spatio-temporal evolution (t >= 10^3), where widely separated zonal jets interact through exponentially weak spatial tails and require long integration horizons before domain size constraints (L << Lambda) intrude.

### Problem Formulation
In this project attempts to solve the same PDE equation using the Physics Informed Neural Network (PINN). The objective is to see weather the PINN is a valid way of predicting the atmospheric flow in 1D by observing the total loss and $L^2$ error values. And attempt to diagnose the issue if the model fails to perform satisfactory.


## Minimal Case Formulation

### Governing PDE
The slow spatio-temporal evolution of the zonally averaged velocity perturbation $U(\eta, \tau) \equiv -A_\eta(\eta, \tau)$ is governed by the non-linear 1D amplitude equation derived in Manfroi & Young (1999). Incorporating linear bottom friction $\mu$, the complete governing partial differential equation (Equation 4.7 in the article) is expressed as:

$$U_\tau = -\mu U - r U_{\eta\eta} - 3 U_{\eta\eta\eta\eta} - 2\gamma(U^2)_{\eta\eta} + \frac{2}{3}(U^3)_{\eta\eta}$$

Setting $\mu = 0$ recovers Equation 4.1 from the article for the inviscid-bottom case:

$$U_\tau = -r U_{\eta\eta} - 3 U_{\eta\eta\eta\eta} - 2\gamma(U^2)_{\eta\eta} + \frac{2}{3}(U^3)_{\eta\eta}$$

where the non-dimensional variables and physical parameters are defined as follows:
* $\eta = \epsilon y$ is the stretched slow meridional spatial coordinate.
* $\tau = \epsilon^4 t$ is the slow temporal scale.
* $U(\eta, \tau)$ represents the leading-order zonal velocity perturbation field.
* $\mu$ is the non-dimensional bottom drag coefficient.
* $\gamma \equiv \beta_1 - \mathfrak{U}_1$ is the effective planetary vorticity gradient parameter combining rotation ($\beta$) and background zonal advection ($\mathfrak{U}$).
* $r \equiv 2 - \gamma^2$ is the negative viscosity coefficient driving linear shear instability when $r > 0$.

### Boundary Conditions
The Boundary condition defined by the paper are:

$$
\begin{aligned}
&(\eta,t)\in[0,100]\times[0,100],\\
&\gamma=1,\qquad r=2-\gamma^2=1,\qquad \mu=0.01,\\[2mm]
&U(0,t)=U(100,t),\\
&U_\eta(0,t)=U_\eta(100,t).
\end{aligned}
$$


### Baseline and Diagnosis
![Diagram 1](actual_pinn_error_time.png)


### Path
Five times as many global updates lowered the residual by a factor of four but
slightly worsened the solution, ruling out insufficient step count as the main
explanation under this optimizer. Naive time marching also failed: the first
window had 0.476 error, the second rose to 0.874 after receiving the preceding
prediction, and the third reached 0.989. Thus, an inaccurate endpoint quickly
contaminated the propagated trajectory.

The oracle restart is not a usable solver because it supplies the true state at
every ten time units, but it is a strong diagnostic. Its 0.193 space-time error
shows that short-domain PINNs can follow the later dynamics when supplied a
correct local initial condition. Oracle-window errors fell to approximately
0.09-0.14 over several late windows after the jets had formed, whereas the
early unstable-growth windows remained the most difficult. The combined result
identifies inaccurate early growth and interface-error amplification as major
causes of the long-time failure. It also gives another counterexample to using
residual as an accuracy certificate: the oracle solution was much more accurate
despite having a larger independently evaluated residual.


## 4. Diagnostic framework: P1–P5

The project uses the required diagnostic kit:

| Diagnostic | Question in this project |
|---|---|
| **P1 — Baseline** | What does the conventional spectral solver predict, and how fast is it? |
| **P2 — Loss components** | How do $L_{\mathrm{PDE}}$, $L_{\mathrm{IC}}$, and $L_{\mathrm{BC}}$ evolve separately? |
| **P3 — Error vs. difficulty knob** | How does relative $L_2$ error change as the time horizon increases? |
| **P4 — Where is the error?** | Where is error concentrated in space-time and Fourier space? |
| **P5 — Cost** | What are training time, inference time, hardware, and timing break-even? |

The principal difficulty knob for P3 is **time horizon**, because the reference field becomes more nonlinear and develops stronger jet amplitudes as time advances.

## 5. P1 — Spectral baseline and standard PINN failure

The Fourier/spectral solver is treated as the reference because the domain is periodic and the PDE is spatially smooth. The standard PINN uses a $2\text{-}64\text{-}64\text{-}64\text{-}1$ tanh network and minimizes

$$\boxed{L = L_{\mathrm{PDE}} + 10L_{\mathrm{IC}} + L_{\mathrm{BC}}}$$

The standard PINN can reduce the optimized loss while remaining physically inaccurate. In the saved paired run used for the formal controls, its final relative $L_2$ error is approximately $0.994$, and it detects only one final eastward jet while the reference has four. This establishes the failure that Part B must diagnose.

![Supervised expressivity control](supervised.png)

**Figure B1. Supervised expressivity control. The same network architecture is trained directly against the spectral reference to separate representational capacity from physics-informed optimization.**

## 6. Part B control 1 — Expressivity vs. physics-informed optimization

The Krishnapriyan-style expressivity control removes the PDE residual and trains the **same $2\text{-}64\text{-}64\text{-}64\text{-}1$ architecture** directly on spectral-reference samples.

Saved-run results:

| Metric | Standard PINN | Supervised same architecture |
|---|---:|---:|
| Final relative $L_2$ | 0.99444 | **0.07185** |
| Final detected jets | 1 | **4** |
| Reference jets | 4 | 4 |
| Supervised training time | — | 9.81 s |

The supervised model is not a PINN; it is a controlled representational test. Its much lower error and correct four-jet count demonstrate that the architecture can represent substantially more of the reference solution than physics-informed training discovers.

$$\boxed{\text{Network expressivity alone does not explain the standard PINN failure.}}$$

This result identifies the physics-informed optimization/training process as a major bottleneck.

## 7. P2 — Loss-component diagnosis

The PINN objective contains three distinct terms:

$$L_{\mathrm{PDE}}, \qquad L_{\mathrm{IC}}, \qquad L_{\mathrm{BC}}$$

These must be monitored separately because a small total loss does not guarantee a small solution error. In the saved Path C benchmark, for example, the standard PINN reached a final PDE loss of approximately $4.01 \times 10^{-4}$ while its final relative field error remained approximately $0.994$. The Fourier PINN reached an even smaller PDE loss of approximately $2.25 \times 10^{-4}$ while its final field error remained approximately $0.973$.

This is an important diagnostic result:

$$\boxed{\text{small PDE residual} \not\Rightarrow \text{accurate physical trajectory.}}$$

Accordingly, all remedies are evaluated against the spectral field using relative $L_2$, jet count, amplitude/energy, and Fourier error rather than training loss alone.

## 8. P3 — Error versus difficulty knob

The time horizon $T$ was selected as the primary difficulty knob. As the system evolves, the spectral reference develops stronger nonlinear zonal jets and increasingly pronounced spatial structure. P3 therefore asks a simple question:

> How does PINN accuracy deteriorate as the required prediction horizon increases?

PINN accuracy is quantified using the relative $L_2$ error,

$$E_{L_2}(T) = \frac{\|U_{\mathrm{PINN}}(\eta,T) - U_{\mathrm{ref}}(\eta,T)\|_2}{\|U_{\mathrm{ref}}(\eta,T)\|_2}.$$

The experiment was evaluated at

$$T = 5, \; 20, \; 50, \; 90, \; 100.$$

### P3.1 Error growth with time horizon

![P3 error versus time horizon](T3.png)

**Figure P3.1** — Relative $L_2$ error versus time horizon. The PINN error increases rapidly with prediction horizon. The error rises from $0.687$ at $T = 5$ to $0.983$ at $T = 20$, and subsequently remains approximately order one. The horizontal dotted line indicates $E_{L_2} = 1$.

The figure identifies a clear transition in PINN performance. At short time, the network retains partial agreement with the reference. By approximately $T = 20$, however, the error is already close to unity. Increasing the time horizon further does not recover accuracy.

### P3.2 Numerical evaluation

| $T$ | Relative $L_2$ error | Jets (Ref / PINN) | Energy Ratio ($E_{\mathrm{PINN}} / E_{\mathrm{ref}}$) |
|:---:|---:|:---:|---:|
| **5** | 0.6866 | 4 / 4 | 0.3496 |
| **20** | 0.9826 | 4 / 5 | 0.0455 |
| **50** | 1.0347 | 4 / 3 | 0.0178 |
| **90** | 1.0211 | 4 / 1 | 0.0410 |
| **100** | 1.0082 | 4 / 1 | 0.0738 |

The numerical results confirm that the principal degradation occurs between the early-time regime and the developed nonlinear state.

At $T = 5$,

$$E_{L_2} = 0.6866,$$

and the PINN still detects all four reference jets. By $T = 20$,

$$E_{L_2} = 0.9826,$$

indicating that the prediction has already reached approximately order-one relative error.

At $T = 50$,

$$E_{L_2} = 1.0347,$$

and only three of the four reference jets are detected. At $T = 100$,

$$E_{L_2} = 1.0082,$$

while the PINN detects only one of the four reference jets.

### P3.3 Evolution of the predicted field

![P3 field snapshots at increasing time horizons](T1.png)

**Figure P3.2** — Reference and PINN fields at increasing difficulty levels. At $T = 5$, the PINN reproduces part of the spatial organization but substantially underestimates the reference amplitudes. As the time horizon increases, the spectral solution develops four strong jets with peak velocities approaching $U \approx 3$, whereas the PINN remains smooth and comparatively low in amplitude.

The field snapshots reveal an important point that is not captured by jet count alone. At $T = 5$, both solutions contain four detected jets, but their amplitudes differ substantially. Therefore,

$$\boxed{\text{correct jet count does not imply an accurate velocity field.}}$$

As $T$ increases, both amplitude accuracy and structural accuracy deteriorate.

### P3.4 Energy and jet-structure collapse

To quantify the amplitude failure, the zonal-flow energy is defined as

$$\frac{1}{2} \int_0^L U(\eta,t)^2 \, d\eta.$$

The energy-recovery ratio is then

$$\frac{E_{\mathrm{PINN}}(t)}{E_{\mathrm{ref}}(t)}.$$

![P3 jet count and energy recovery](T2.png)

**Figure P3.3** — Jet-count and energy recovery as difficulty increases. The left panel shows progressive deterioration of the PINN jet structure. The right panel shows a severe collapse in predicted energy relative to the spectral reference.

The energy diagnostic reveals an even stronger failure than the relative $L_2$ metric alone. At $T = 5$,

$$\frac{E_{\mathrm{PINN}}(5)}{E_{\mathrm{ref}}(5)} = 0.3496,$$

so the PINN contains only approximately $35\%$ of the reference energy even though it detects four jets.

By $T = 20$, the energy ratio falls to

$$0.0455,$$

and at $T = 50$,

$$0.0178,$$

meaning that the PINN contains only approximately $1.8\%$ of the reference zonal-flow energy.

The failure is therefore not simply an incorrect jet count. It is primarily characterized by severe amplitude and energy underprediction, followed by deterioration of the spatial jet structure.

### P3.5 Main finding

P3 identifies the transition from the early-time solution to the developed nonlinear jet state as the principal difficulty regime:

$$\boxed{
\text{Increasing time horizon}
\rightarrow
\text{amplitude/energy collapse}
\rightarrow
\text{loss of jet structure}
\rightarrow
\mathcal{O}(1)\text{ relative error.}
}$$

The standard PINN therefore performs partially at short horizons but does not sustain the nonlinear growth exhibited by the spectral reference.

This result motivates P4, which examines where the remaining error is concentrated in space, time, and Fourier frequency. It also motivates the controlled experiments in Part B, which test whether the observed failure is caused primarily by insufficient network expressivity, temporal causality, or difficulty optimizing the physics-informed objective.

## 9. P4 — Space-time and frequency diagnosis

P4 asks **where** the PINN error occurs. Two complementary diagnostics are used:

1. **Space-time error:** large structured errors appear where the reference develops strong jet ridges, showing that the discrepancy is associated with missing physical structures rather than random numerical noise.
2. **Fourier error:** the spatial error

   $$e(\eta,t) = U_{\mathrm{PINN}}(\eta,t) - U_{\mathrm{ref}}(\eta,t)$$

   is transformed to $\widehat{e}(k,t)$, and modal error energy $|\widehat{e}(k,t)|^2$ is inspected.

The frequency analysis supports the conclusion that spatial-frequency representation contributes to the difficulty, but it does **not** establish high frequency as the sole cause. The PINN also exhibits severe amplitude/energy underprediction, and causal training does not remove the failure.

![Causality control](konan.png)

**Figure B2. Causality control: full-domain standard PINN versus sequential time-window PINN.**

## 10. Part B control 2 — Causality

To test whether full-domain training is the dominant cause of late-time failure, the same standard architecture is trained sequentially over

$$[0, 20] \rightarrow [20, 40] \rightarrow [40, 60] \rightarrow [60, 80] \rightarrow [80, 100]$$

The corrected PDE, optimizer family, loss form, and periodic boundary conditions are retained; the principal experimental change is temporal time-marching.

Saved-run results:

| Metric | Standard PINN | Causal PINN |
|---|---:|---:|
| Final relative $L_2$ | **0.99515** | **1.01405** |
| Final jets | 1 | 1 |
| Reference jets | 4 | 4 |
| Training time | 170.56 s | 311.72 s |

The causal model does not reduce the final error; it is slightly worse in this run. Therefore,

$$\boxed{\text{causality alone is not supported as the dominant failure mechanism.}}$$

This does not prove causality is irrelevant. It shows that simple sequential PINN-to-PINN handoff is insufficient to recover the missing high-amplitude jet dynamics.

## 11. Part B root-cause conclusion

The controlled evidence narrows the diagnosis:

$$\text{Standard PINN fails} \rightarrow \text{same architecture succeeds supervised} \rightarrow \text{expressivity alone is insufficient as an explanation}$$

and

$$\text{causal time-marching still fails} \rightarrow \text{causality alone is not the dominant explanation}.$$

P4 additionally shows that frequency representation matters, but the persistent amplitude/energy deficit means classical spectral bias alone is also insufficient as a complete explanation.

The most defensible conclusion is therefore:

$$\boxed{\text{physics-informed optimization of the high-order nonlinear PDE is the principal remaining bottleneck.}}$$

This diagnosis motivates a targeted representation change in Part C.

# Part C — Targeted remedy: Fourier-feature PINN

## 12. Why Fourier features?

The domain is periodic and the reference solution is multimode. The Fourier-feature PINN replaces the raw spatial coordinate with eight periodic sine/cosine pairs,

$$\sin(2\pi m\eta/L), \qquad \cos(2\pi m\eta/L), \qquad m=1,\ldots,8$$

plus normalized time. The resulting 17-input representation is designed to expose periodic spatial structure directly to the optimizer. Importantly, the improved model remains a PINN: the spectral reference is not used as a training target.

![Saved Path C paired run](pathC_summary.png)

**Figure C1. Saved paired Path C benchmark: standard PINN versus Fourier-feature PINN on the same reference and training setup.**

## 13. Saved paired Path C result

In the saved same-run benchmark, Fourier features reduce the final relative $L_2$ error modestly:

$$0.99444 \rightarrow 0.97347$$

The final PDE loss also decreases from approximately $4.01 \times 10^{-4}$ to $2.25 \times 10^{-4}$. However, the physical jet-count criterion does not improve in that particular final-time run: reference/standard/Fourier = $4/1/0$.

This result is therefore **partial and negative** at final time: Fourier features improve the optimized numerical quantities but do not recover the required physical jet structure.

![Dynamic Fourier comparison at t=90](kouassi.png)

**Figure C2. Separate dynamic Fourier-feature run at t=90. This run is reported separately from the saved final-time P5 benchmark.**

## 14. Dynamic Fourier-feature evidence

A later dynamic run provides additional time-local evidence. Because it is a separate run, its values are not substituted into the saved P5 benchmark.

At $t = 50$:

$$L_{2,\mathrm{std}} = 1.037, \qquad L_{2,\mathrm{Fourier}} = 0.910$$

an approximately **12.2%** reduction, with jet counts reference/standard/Fourier = $4/3/4$.

At $t = 90$:

$$L_{2,\mathrm{std}} = 1.020, \qquad L_{2,\mathrm{Fourier}} = 0.941$$

an approximately **7.7%** reduction, with jet counts $4/1/3$.

These snapshots show that Fourier encoding can preserve more of the jet organization at intermediate and late times, but the benefit is time-dependent and weakens as the nonlinear evolution proceeds. The Fourier prediction still strongly underestimates jet amplitudes.

$$\boxed{\text{Fourier features improve structure partially, but do not solve the field-amplitude failure.}}$$

![Fourier error comparison](pathC_fourier_error_before_after.png)

**Figure C3. Fourier-space error comparison for the standard and Fourier-feature PINNs.**

## 15. Frequency interpretation

The Fourier-feature experiment is useful precisely because it is a targeted test of the P4 frequency diagnosis. The partial improvement supports the hypothesis that periodic/multiscale representation contributes to the failure. However, because the error remains large and the jet amplitudes remain too small, the evidence does not support the stronger claim that high-frequency content is the sole cause.

The fourth-order term $-3 U_{\eta\eta\eta\eta}$ should also be interpreted carefully. It strongly damps high wavenumbers in the physical PDE, but it makes automatic-differentiation residuals sensitive to frequency because a Fourier mode is multiplied by $k^4$ under four spatial derivatives.

![3D zonal-jet evolution comparison](3D_comparison_all_models.png)

**Figure C4. Three-dimensional comparison of the spectral reference and physics-informed approximations.**

![3D pointwise error comparison](3D_error_comparison.png)

**Figure C5. Structured pointwise error is concentrated around the jet structures that the PINNs fail to reproduce.**

## 16. Secondary remedy — gradual-diffusion curriculum

A secondary experiment temporarily adds artificial diffusion and reduces it progressively to zero while retaining network weights between stages. The intent is to begin from a smoother optimization problem and gradually restore the original PDE.

The curriculum changes the predicted field but does not convincingly recover the strong reference amplitudes. It is therefore retained as a useful negative/secondary result rather than the primary Path C remedy.

## 17. P5 — Computational cost

The saved P5 benchmark times all methods on the **same physical horizon, $t = 0 \ldots 100$** and on the same machine.

**Hardware:** Windows 11; AMD64 Family 25 Model 124; PyTorch CPU; no GPU; PyTorch 2.14.1+cpu.

| Method / operation | Wall time |
|---|---:|
| Spectral forward solve | **0.1733 s** |
| Standard PINN training | **146.08 s** |
| Fourier PINN training | **291.34 s** |
| Standard PINN inference, all saved frames | **0.00324 ± 0.00105 s** |
| Fourier PINN inference, all saved frames | **0.00491 ± 0.00062 s** |

For a single forward query,

$$T_{\mathrm{train}} \gg T_{\mathrm{sol}},$$

so the spectral method is decisively cheaper. A timing-only surrogate break-even estimate is

$$N^\star = \frac{T_{\mathrm{train}}}{T_{\mathrm{sol}} - T_{\mathrm{inf}}},$$

giving approximately $N^\star_{\mathrm{std}} \approx 859$ and $N^\star_{\mathrm{Fourier}} \approx 1730$. These are **not practical break-even claims**, because the PINNs in this benchmark are substantially less accurate than the spectral solution. A fast surrogate is not valuable if it fails the physical accuracy requirement.

## 18. Overall assessment

| Evidence | Finding | Diagnostic implication |
|---|---|---|
| P3 time-horizon sweep | $L_2 : 0.687$ at $T = 5 \rightarrow$ order one for $T \ge 20$; energy ratio falls to $0.0178$ at $T = 50$ | Failure emerges as nonlinear jet amplitude/energy develops |
| Standard PINN | Order-one final field error; jet amplitudes collapse | Baseline failure reproduced |
| Supervised same architecture | Final $L_2 \approx 0.0719$, 4 jets | Architecture can represent the reference much better |
| Loss components | Small PDE loss can coexist with large field error | Training loss alone is insufficient |
| Causal time windows | Final relative $L_2$: $0.995 \rightarrow 1.014$ (1.9% worsening); final jets = 4/1/1 (reference/standard/causal) | Time-marching does not improve the solution; causality is not the dominant failure mechanism. |
| Fourier features | Partial/time-dependent improvement | Frequency representation contributes but is not sufficient |
| Gradual diffusion | No convincing amplitude recovery | Smoothing curriculum alone is insufficient |
| P5 | Spectral solve $0.173\text{ s}$ vs minutes of PINN training | Spectral method dominates this forward problem |

### 19. Final technical conclusion
The experiments do not support a single-cause explanation such as “high frequency” or “causality.” The strongest controlled result is the supervised expressivity test: the same architecture can fit the reference much more accurately than the physics-informed training discovers. Causal time marching does not remove the failure, and Fourier features help only partially. The remaining bottleneck is therefore best described as **physics-informed optimization of a high-order nonlinear PDE, with frequency representation and long-time amplitude growth contributing to the difficulty**.

## 20. Assumptions, limitations, and reproducibility

- The spherical visualization used elsewhere in the project is illustrative only; the governing PDE is one-dimensional and periodic.
- Results from different training runs are labeled separately. The saved P5 benchmark values are not numerically mixed with the later \(t=50\)/\(t=90\) dynamic Fourier run.
- The formal P3 sweep was generated using `03_P3_error_vs_time_horizon_CORRECTED.py` at $T = 5, 20, 50, 90, 100$. The corresponding numerical results and physical diagnostics are reported in Section 8.
- The corrected causality control is implemented in `07_corrected_causality_control_PINN.py`.
- The corrected cubic coefficient is \(2/3\) in all final physics-based scripts.
- PINN results depend on random initialization, collocation sampling, optimizer settings, and hardware. Repeated-seed statistics would strengthen the conclusions.
- The present conclusions apply to this reduced forward problem and should not be generalized to all PINN applications.

### Key reproducibility files
- `01_standard_PINN_baseline_CORRECTED.py`
- `02_supervised_expressivity_control_CORRECTED.py`
- `03_P3_error_vs_time_horizon_CORRECTED.py`
- `07_corrected_causality_control_PINN.py`
- `02_fourier_feature_PINN_and_P5_CORRECTED.py`

### References
- Huang, H.-P., & Robinson, W. A. (1998). *Two-Dimensional Turbulence and Persistent Zonal Jets in a Global Barotropic Model.*
- Manfroi, A. J., & Young, W. R. (1999). Reduced-model analysis of zonal-jet dynamics and slow evolution near instability onset.
- Krishnapriyan et al. (2021). PINN failure-mode analysis motivating the supervised expressivity and curriculum-style diagnostic logic used here.